# Day 2 — Notebook 05  
# End-to-End RAG, Evidence, Hallucination & Responsible Output

**Hands-on outcome:** Build the RAG assistant, make retrieved evidence visible before generation, compare an irresponsible prompt with a responsible grounded prompt, and show context/hybrid-search behavior using DataFrames.

> **** “To see exactly what evidence the model received before judging the answer.”

## 1. Load Azure endpoints and the Day 1 vector database

In [ ]:
from pathlib import Path
import os, json, re, time, math
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

ROOT = Path(".")
POLICY_DIR = ROOT / "data" / "healthcare_policies"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

load_dotenv(".env", override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model = os.getenv("AZURE_OPENAI_MODEL")
embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

if not all([endpoint, api_key, model, embedding_model]):
    raise ValueError(
        "Missing Azure configuration. Check AZURE_OPENAI_ENDPOINT, "
        "AZURE_OPENAI_API_KEY, AZURE_OPENAI_MODEL and "
        "AZURE_OPENAI_EMBEDDING_MODEL in .env"
    )

client = OpenAI(base_url=endpoint, api_key=api_key)

env_df = pd.DataFrame([
    {"Component":"Chat / Generation", "Azure deployment":model, "Purpose":"Generate grounded answers / rerank"},
    {"Component":"Embeddings", "Azure deployment":embedding_model, "Purpose":"Convert text and queries into vectors"}
])
display(env_df)

import chromadb

chroma_client = chromadb.PersistentClient(path=str(ARTIFACT_DIR / "chroma_policy_db"))
section_collection = chroma_client.get_collection("policy_section")

def query_vector_db(question, top_k=4, where=None):
    qvec = client.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    return section_collection.query(
        query_embeddings=[qvec],
        n_results=top_k,
        where=where,
        include=["documents","metadatas","distances"]
    )

## 2. Retrieval — always show the evidence table first

In [ ]:
def retrieve_chunks(question, top_k=4, plan_type=None):
    where = {"plan_type":plan_type} if plan_type else None
    raw = query_vector_db(question, top_k=top_k, where=where)

    chunks = []
    for rank,(text,meta,distance) in enumerate(
        zip(raw["documents"][0],raw["metadatas"][0],raw["distances"][0]), start=1
    ):
        chunks.append({
            "rank":rank,
            "text":text,
            "metadata":meta,
            "distance":distance,
            "similarity":1-distance,
            "citation":f"{meta['doc_id']} | {meta['section']} | p.{meta['page']}"
        })
    return chunks

def evidence_df(chunks):
    return pd.DataFrame([
        {
            "Rank":c["rank"],
            "Similarity ↑":round(c["similarity"],4),
            "Document":c["metadata"]["doc_id"],
            "Plan":c["metadata"]["plan_type"],
            "Section":c["metadata"]["section"],
            "Page":c["metadata"]["page"],
            "Citation":c["citation"],
            "Evidence":re.sub(r"\s+"," ",c["text"])[:260] + "..."
        }
        for c in chunks
    ])

question = "For a Gold PPO member, does the 11th physical therapy visit require prior authorization?"
chunks = retrieve_chunks(question, top_k=4, plan_type="Gold PPO")
display(evidence_df(chunks))

> **Takeaway:** Before reading the generated answer, inspect whether the correct evidence is actually present. This separates retrieval failure from generation failure.

## 3. Assemble citation-aware context and generate a grounded answer

In [ ]:
def assemble_context(chunks):
    return "\n\n".join(
        f"[SOURCE {i}: {c['citation']}]\n{c['text']}"
        for i,c in enumerate(chunks,start=1)
    )

RESPONSIBLE_SYSTEM = """
You are a healthcare payer policy assistant.
Use only the supplied policy context.
If the context does not contain enough evidence, say exactly: Insufficient information.
Do not use outside knowledge and do not infer missing benefit rules.
Cite the supporting source labels exactly as [SOURCE n].
Keep the answer concise and operational.
"""

def generate_answer(question, chunks, system_prompt=RESPONSIBLE_SYSTEM):
    context = assemble_context(chunks)
    start = time.perf_counter()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":f"POLICY CONTEXT:\n{context}\n\nQUESTION:\n{question}"}
        ],
        temperature=0
    )

    latency = time.perf_counter()-start
    return response, latency

response, latency = generate_answer(question, chunks)

answer_summary = pd.DataFrame([{
    "Question":question,
    "Answer":response.choices[0].message.content,
    "Evidence Chunks":len(chunks),
    "Prompt Tokens":response.usage.prompt_tokens,
    "Completion Tokens":response.usage.completion_tokens,
    "Latency Seconds":round(latency,2)
}])
display(answer_summary)

The output now connects **business answer + evidence volume + token consumption + latency**.

> **Takeaway:** The answer is the business output; the retrieval evidence and telemetry are the engineering/governance output.

## 4. Reusable Claims & Benefits Assistant with explainability

In [ ]:
def infer_plan_filter(question):
    q = question.lower()
    if "gold ppo" in q: return "Gold PPO"
    if "silver hmo" in q: return "Silver HMO"
    return None

def claims_benefits_assistant(question, top_k=4, system_prompt=RESPONSIBLE_SYSTEM):
    plan_filter = infer_plan_filter(question)
    chunks = retrieve_chunks(question, top_k=top_k, plan_type=plan_filter)
    response, latency = generate_answer(question, chunks, system_prompt)

    return {
        "question":question,
        "plan_filter":plan_filter,
        "answer":response.choices[0].message.content,
        "retrieved":chunks,
        "usage":{
            "prompt_tokens":response.usage.prompt_tokens,
            "completion_tokens":response.usage.completion_tokens,
            "total_tokens":response.usage.total_tokens
        },
        "latency":latency
    }

demo = claims_benefits_assistant("How many chiropractic visits are covered under Silver HMO?")
display(evidence_df(demo["retrieved"]))
display(pd.DataFrame([{
    "Plan Filter":demo["plan_filter"],
    "Answer":demo["answer"],
    "Total Tokens":demo["usage"]["total_tokens"],
    "Latency Seconds":round(demo["latency"],2)
}]))

## 5. Hallucination demonstration — irresponsible vs responsible output

The corpus intentionally does **not** contain a Gold PPO annual acupuncture limit.

We will ask the same model twice:
- **Irresponsible instruction:** infer a likely answer when policy evidence is missing.
- **Responsible instruction:** answer only from supplied evidence and refuse unsupported facts.

This is intentionally a teaching demonstration of a bad design pattern — not a recommended production prompt.

In [ ]:
hallucination_question = "What is the annual acupuncture visit limit under the Gold PPO plan?"
hallucination_chunks = retrieve_chunks(
    hallucination_question, top_k=4, plan_type="Gold PPO"
)

display(evidence_df(hallucination_chunks))

In [ ]:
IRRESPONSIBLE_SYSTEM = """
You are a confident healthcare benefits assistant.
Answer the user's question even if the supplied context is incomplete.
If an exact value is absent, infer a plausible industry-standard value.
Do not mention uncertainty.
"""

bad_response, bad_latency = generate_answer(
    hallucination_question, hallucination_chunks, IRRESPONSIBLE_SYSTEM
)
good_response, good_latency = generate_answer(
    hallucination_question, hallucination_chunks, RESPONSIBLE_SYSTEM
)

responsibility_df = pd.DataFrame([
    {
        "Behavior":"Irresponsible / hallucination-prone",
        "Evidence Rule":"May infer missing facts",
        "Answer":bad_response.choices[0].message.content,
        "Expected Risk":"Unsupported policy claim"
    },
    {
        "Behavior":"Responsible / grounded",
        "Evidence Rule":"Use supplied evidence only",
        "Answer":good_response.choices[0].message.content,
        "Expected Risk":"Controlled refusal / escalation"
    }
])
display(responsibility_df)

> **Takeaway:** Hallucination is not only a model problem. Prompting, retrieval scope and failure policy all influence whether unsupported claims reach the user.

### Responsible enterprise behavior
When policy evidence is absent, the correct output is often:
**“Insufficient information” + source/evidence gap + escalation path**, not a guessed number.

## 6. Context explosion — make the trade-off measurable

In [ ]:
question = "Does prior authorization guarantee claim payment for Gold PPO?"

context_rows = []
for k in [2,4,8]:
    result = claims_benefits_assistant(question, top_k=k)
    context_rows.append({
        "Top-K":k,
        "Retrieved Chunks":len(result["retrieved"]),
        "Unique Documents":len(set(c["metadata"]["doc_id"] for c in result["retrieved"])),
        "Context Characters":sum(len(c["text"]) for c in result["retrieved"]),
        "Prompt Tokens":result["usage"]["prompt_tokens"],
        "Total Tokens":result["usage"]["total_tokens"],
        "Latency Seconds":round(result["latency"],2),
        "Answer":result["answer"]
    })

display(pd.DataFrame(context_rows))

> **Takeaway:** Context explosion is visible as increased chunks, characters and prompt tokens. More context is useful only if the additional evidence improves answer quality.

## 7. Hybrid retrieval — show lexical rank, vector rank and fused rank

In [ ]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

records = load_jsonl(ARTIFACT_DIR / "policy_chunks_section.jsonl")
record_by_id = {r["chunk_id"]:r for r in records}

def lexical_rank(question, top_n=20):
    terms = set(re.findall(r"[a-z0-9]+", question.lower()))
    scored = []
    for r in records:
        tokens = set(re.findall(r"[a-z0-9]+",r["text"].lower()))
        scored.append((len(terms & tokens),r["chunk_id"]))
    scored.sort(key=lambda x:x[0],reverse=True)
    return [cid for score,cid in scored[:top_n] if score>0]

def vector_rank(question, top_n=20):
    raw = query_vector_db(question,top_k=top_n)
    return raw["ids"][0]

def rrf(rank_lists,k=60):
    scores={}
    for rank_list in rank_lists:
        for rank,doc_id in enumerate(rank_list,start=1):
            scores[doc_id]=scores.get(doc_id,0)+1/(k+rank)
    return sorted(scores.items(),key=lambda x:x[1],reverse=True)

query = "How long after a claim denial can a provider dispute the decision?"
lex = lexical_rank(query)
vec = vector_rank(query)
fused = rrf([lex,vec])[:6]

hybrid_rows = []
for fused_rank,(cid,score) in enumerate(fused,start=1):
    r = record_by_id[cid]
    hybrid_rows.append({
        "Fused Rank":fused_rank,
        "Lexical Rank":lex.index(cid)+1 if cid in lex else None,
        "Vector Rank":vec.index(cid)+1 if cid in vec else None,
        "RRF Score":round(score,5),
        "Document":r["doc_id"],
        "Section":r["section"],
        "Text":re.sub(r"\s+"," ",r["text"])[:220]+"..."
    })

display(pd.DataFrame(hybrid_rows))

> **Takeaway:** Hybrid retrieval gives the ranking system two signals: exact lexical evidence and semantic meaning.

## Day 2 checkpoint

You can now visually distinguish:

**retrieval evidence → grounded generation → responsible failure behavior → context trade-offs → hybrid ranking**

The next notebook asks the production question:

**How do we measure whether the system is actually reliable across many questions?**